In [5]:
"""
Contact-angle measurement from YOLO segmentation masks.
Generates Raw, Mask segmnentation, and Final Annotated images for thesis visualization.
"""

import glob
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ultralytics import YOLO

# Configuration & Paths
model_path = r'C:\Users\nahid.salimi\Downloads\Droplet_Contact_Angle_Code\best.pt'
model = YOLO(model_path)

input_folder = r'C:\Users\nahid.salimi\OneDrive\Fotos'
output_folder = os.path.join(input_folder, 'Annotated_Results')
os.makedirs(output_folder, exist_ok=True)

# Quality Control Threshold
RMSE_FLAG_THRESHOLD = 4.0  # px


def _contact_angle_from_slope(slope, into_drop_dx_sign):
    """Calculates the contact angle using direction vector dot product."""
    v_tangent = np.array([into_drop_dx_sign, slope * into_drop_dx_sign], dtype=float)
    v_tangent /= np.linalg.norm(v_tangent)
    v_baseline_into_drop = np.array([into_drop_dx_sign, 0.0])
    cos_angle = np.clip(np.dot(v_tangent, v_baseline_into_drop), -1.0, 1.0)
    return float(np.degrees(np.arccos(cos_angle)))


def fit_droplet_robust_tangents(contour_points):
    """
    Fits tangent lines to droplet flanks with Adaptive Baseline Optimization.
    Automatically selects the baseline percentile that yields the minimum RMSE.
    """
    pts = contour_points.reshape(-1, 2).astype(np.float32)

    if len(pts) < 10:
        return None

    sorted_y = np.sort(pts[:, 1])

    # --- ADAPTIVE BASELINE SEARCH (82% to 88%) ---
    best_res = None
    min_rmse = float('inf')

    # Sweep through standard baseline percentiles
    for percentile in range(82, 90, 2):
        base_y = np.percentile(sorted_y, percentile)

        # Contact Points Estimation
        near_base = pts[np.abs(pts[:, 1] - base_y) < 5]
        if len(near_base) >= 2:
            x_min = np.min(near_base[:, 0])
            x_max = np.max(near_base[:, 0])
        else:
            x_min = np.min(pts[:, 0])
            x_max = np.max(pts[:, 0])

        width = x_max - x_min
        FIT_GAP_PX = 2.0
        MAX_HEIGHT_OFFSET = width * 0.35

        # Flank Masks
        left_mask = (
            (pts[:, 0] >= x_min)
            & (pts[:, 0] <= x_min + width * 0.10)
            & (pts[:, 1] <= base_y - FIT_GAP_PX)
            & (pts[:, 1] >= base_y - MAX_HEIGHT_OFFSET)
        )

        right_mask = (
            (pts[:, 0] >= x_max - width * 0.10)
            & (pts[:, 0] <= x_max)
            & (pts[:, 1] <= base_y - FIT_GAP_PX)
            & (pts[:, 1] >= base_y - MAX_HEIGHT_OFFSET)
        )

        left_pts = pts[left_mask]
        right_pts = pts[right_mask]

        if len(left_pts) < 3 or len(right_pts) < 3:
            continue

        # Left Linear Fit
        poly_l = np.polyfit(left_pts[:, 0], left_pts[:, 1], 1)
        slope_l = poly_l[0]
        y_pred_l = poly_l[0] * left_pts[:, 0] + poly_l[1]
        rmse_l = np.sqrt(np.mean((left_pts[:, 1] - y_pred_l) ** 2))

        # Right Linear Fit
        poly_r = np.polyfit(right_pts[:, 0], right_pts[:, 1], 1)
        slope_r = poly_r[0]
        y_pred_r = poly_r[0] * right_pts[:, 0] + poly_r[1]
        rmse_r = np.sqrt(np.mean((right_pts[:, 1] - y_pred_r) ** 2))

        avg_rmse = (rmse_l + rmse_r) / 2.0

        # Update if this percentile achieves a better (lower) RMSE
        if avg_rmse < min_rmse:
            min_rmse = avg_rmse
            angle_l = _contact_angle_from_slope(slope_l, into_drop_dx_sign=+1)
            angle_r = _contact_angle_from_slope(slope_r, into_drop_dx_sign=-1)
            pt_left = (int(x_min), int(base_y))
            pt_right = (int(x_max), int(base_y))

            best_res = (
                angle_l,
                angle_r,
                pt_left,
                pt_right,
                slope_l,
                slope_r,
                int(base_y),
                avg_rmse,
                percentile,
            )

    return best_res


# Fetch images
image_paths = []
for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp']:
    image_paths.extend(glob.glob(os.path.join(input_folder, ext)))

results_data = []

for img_path in image_paths:
    file_name = os.path.basename(img_path)
    base_name, ext = os.path.splitext(file_name)

    results = model.predict(
        source=img_path,
        conf=0.25,
        retina_masks=True,
        verbose=False,
    )

    for result in results:
        img = cv2.imread(img_path)

        if result.masks is None:
            print(f"{file_name}: no mask detected, skipping.")
            continue

        masks = result.masks.xy[0].astype(np.int32)

        fit_res = fit_droplet_robust_tangents(masks)
        if fit_res is None:
            print(f"{file_name}: not enough contour points to fit, skipping.")
            continue

        (
            deg_l,
            deg_r,
            pt_l,
            pt_r,
            slope_l,
            slope_r,
            base_y,
            rmse,
            opt_percentile,
        ) = fit_res
        avg_angle = (deg_l + deg_r) / 2.0
        flag = (
            "OK"
            if (RMSE_FLAG_THRESHOLD is None or rmse <= RMSE_FLAG_THRESHOLD)
            else "CHECK MANUALLY"
        )

        results_data.append({
            'Filename': file_name,
            'Left Angle (deg)': round(deg_l, 2),
            'Right Angle (deg)': round(deg_r, 2),
            'Average Angle (deg)': round(avg_angle, 2),
            'Fit RMSE (px)': round(rmse, 3),
            'Opt Percentile': opt_percentile,
            'Flag': flag,
        })
        print(
            f"{file_name}: L={deg_l:.2f} R={deg_r:.2f} Avg={avg_angle:.2f} "
            f"RMSE={rmse:.3f} P={opt_percentile}% [{flag}]"
        )

        # ----------------------------------------------------
        # FIXED: SAVE IMAGE (b) - SEGMENTATION-MASK OVERLAY
        # ----------------------------------------------------
        mask_visualization_img = img.copy()
        cv2.polylines(mask_visualization_img, [masks], isClosed=True, color=(0, 255, 0), thickness=2)
        mask_output_path = os.path.join(output_folder, f"{base_name}_mask{ext}")
        cv2.imwrite(mask_output_path, mask_visualization_img)
        # ----------------------------------------------------

        # 1. Draw Mask Contour (Green)
        cv2.polylines(img, [masks], isClosed=True, color=(0, 255, 0), thickness=2)

        # 2. Baseline (White)
        cv2.line(
            img,
            (pt_l[0] - 25, base_y),
            (pt_r[0] + 25, base_y),
            (255, 255, 255),
            2,
        )

        # 3. Accurate Tangent Drawing
        tangent_len = 50

        rad_l = np.radians(deg_l)
        pt_l_end = (
            int(pt_l[0] + tangent_len * np.cos(rad_l)),
            int(pt_l[1] - tangent_len * np.sin(rad_l)),
        )
        cv2.line(img, pt_l, pt_l_end, (0, 0, 255), 2)

        rad_r = np.radians(deg_r)
        pt_r_end = (
            int(pt_r[0] - tangent_len * np.cos(rad_r)),
            int(pt_r[1] - tangent_len * np.sin(rad_r)),
        )
        cv2.line(img, pt_r, pt_r_end, (0, 0, 255), 2)

        # 4. Text Labels
        text_color = (255, 255, 0) if flag == "OK" else (0, 0, 255)
        cv2.putText(
            img, f'L: {deg_l:.1f} deg', (pt_l[0] - 80, base_y - 12),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, text_color, 2,
        )
        cv2.putText(
            img, f'R: {deg_r:.1f} deg', (pt_r[0] + 10, base_y - 12),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, text_color, 2,
        )
        cv2.putText(
            img, f'RMSE: {rmse:.2f}px [{flag}]', (pt_l[0], base_y + 25),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1,
        )

        # Save Final Output Image (c)
        output_img_path = os.path.join(output_folder, file_name)
        cv2.imwrite(output_img_path, img)

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(6, 4))
        plt.imshow(img_rgb)
        plt.title(
            f'{file_name} | Avg: {avg_angle:.1f}\u00b0 | RMSE: {rmse:.2f}px [{flag}]'
        )
        plt.axis('off')
        plt.savefig(os.path.join(output_folder, f"plot_{file_name}"))
        plt.close()

# Save CSV Summary
csv_save_path = os.path.join(input_folder, 'contact_angles_summary.csv')
df = pd.DataFrame(results_data)
df.to_csv(csv_save_path, index=False)
print(f"\nDone. {len(results_data)} images processed successfully.")
print(f"Results saved to {csv_save_path}")

11_droplet.png: L=26.89 R=34.39 Avg=30.64 RMSE=0.577 P=82% [OK]
13_droplet.png: L=41.02 R=42.51 Avg=41.76 RMSE=0.631 P=82% [OK]
14_droplet.png: L=46.86 R=46.99 Avg=46.92 RMSE=0.742 P=82% [OK]
16_droplet.png: L=35.97 R=36.72 Avg=36.34 RMSE=0.383 P=82% [OK]
B3_1_droplet.png: L=24.62 R=20.41 Avg=22.51 RMSE=0.799 P=82% [OK]
B3_2_droplet.png: L=27.53 R=26.39 Avg=26.96 RMSE=0.505 P=82% [OK]
B3_droplet.png: L=31.94 R=33.66 Avg=32.80 RMSE=0.561 P=82% [OK]

Done. 7 images processed successfully.
Results saved to C:\Users\nahid.salimi\OneDrive\Fotos\contact_angles_summary.csv
